# Vanilla CNN — Hyperparameter Random Search
## Forest Cover Semantic Segmentation · 256×256 px

Runs a **random search** over a vanilla encoder–decoder CNN.  
Architecture: `Conv2D ▶ MaxPool ▶ UpSampling` (no skip connections, no batch norm)  
Primary metric: **Val IoU (Jaccard)**

---
### 📋 Instructions
1. Upload your `images/` and `masks/` folders to Google Drive under  
   `My Drive / DNN-Project / Kalana /`
2. Update **`DRIVE_BASE`** in the *Configuration* cell below if your path differs
3. Set runtime to GPU: `Runtime → Change runtime type → T4 GPU`
4. Run all cells: `Runtime → Run all`

**Outputs saved back to Drive**
| File | Contents |
|---|---|
| `best_vanilla_cnn.keras` | Best model weights |
| `hyperparam_search_results.csv` | All trials ranked by val IoU |
| `best_hyperparams.txt` | Conference table + best config |

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout[:900])
else:
    print("[WARNING] No GPU detected.")
    print("Go to: Runtime → Change runtime type → T4 GPU")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, csv, time, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ──────────────────────────────────────────────────────────────────────────────
# UPDATE THIS PATH to match your Google Drive folder structure
DRIVE_BASE   = "/content/drive/MyDrive/DNN-Project/Kalana"
# ──────────────────────────────────────────────────────────────────────────────

IMAGE_FOLDER = os.path.join(DRIVE_BASE, "images")
MASK_FOLDER  = os.path.join(DRIVE_BASE, "masks")
OUTPUT_DIR   = DRIVE_BASE
IMG_SIZE     = 256

N_TRIALS     = 20    # ← increase for a more thorough search
MAX_EPOCHS   = 30    # EarlyStopping will cut most runs far shorter
RANDOM_SEED  = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU')}")
print(f"Images path : {IMAGE_FOLDER}  → exists={os.path.isdir(IMAGE_FOLDER)}")
print(f"Masks path  : {MASK_FOLDER}   → exists={os.path.isdir(MASK_FOLDER)}")
print(f"Output dir  : {OUTPUT_DIR}")

In [ ]:
PARAM_GRID = {
    # log-spaced learning rates covering two decades
    "learning_rate": [1e-4, 3e-4, 5e-4, 1e-3, 2e-3, 5e-3],
    "batch_size":    [8, 16, 32],
    "base_filters":  [16, 32, 64],   # filters in 1st encoder stage; doubles each stage
    "kernel_size":   [3, 5],
    "depth":         [2, 3],         # encoder conv-pool stages
    "dropout_rate":  [0.0, 0.1, 0.2, 0.3],
    "optimizer_name":["adam", "rmsprop"],
    "loss_fn":       ["binary_crossentropy", "dice", "bce_dice"],
}

total = 1
print("Search space:")
for k, v in PARAM_GRID.items():
    print(f"  {k:<20} {v}")
    total *= len(v)
print(f"\nTotal combinations : {total:,}")
print(f"Random trials       : {N_TRIALS}")

In [ ]:
# ── Loss functions ────────────────────────────────────────────────────────────

def dice_loss(y_true, y_pred):
    """Soft Dice loss — better for imbalanced binary masks."""
    smooth   = 1e-6
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1.0 - (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )

def bce_dice_loss(y_true, y_pred):
    """BCE + Dice combo: drives pixel accuracy and region overlap simultaneously."""
    bce = tf.keras.losses.binary_crossentropy(
        tf.reshape(y_true, [-1]), tf.reshape(y_pred, [-1])
    )
    return bce + dice_loss(y_true, y_pred)

LOSS_MAP = {
    "binary_crossentropy": "binary_crossentropy",
    "dice":                dice_loss,
    "bce_dice":            bce_dice_loss,
}

# ── Streaming IoU metric ──────────────────────────────────────────────────────

class BinaryIoU(tf.keras.metrics.Metric):
    """Streaming binary IoU at threshold 0.5; accumulates across batches."""

    def __init__(self, threshold=0.5, name="iou", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold     = threshold
        self._intersection = self.add_weight(name="intersection", shape=(), initializer="zeros")
        self._union        = self.add_weight(name="union",        shape=(), initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_bin   = tf.cast(y_pred >= self.threshold, tf.float32)
        y_true_f     = tf.cast(y_true, tf.float32)
        intersection = tf.reduce_sum(y_true_f * y_pred_bin)
        union        = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_bin) - intersection
        self._intersection.assign_add(intersection)
        self._union.assign_add(union + 1e-6)

    def result(self):
        return self._intersection / self._union

    def reset_state(self):
        self._intersection.assign(0.0)
        self._union.assign(0.0)

print("✓ Loss functions and BinaryIoU metric ready.")

In [ ]:
def build_vanilla_cnn(base_filters, kernel_size, depth,
                      dropout_rate, learning_rate, optimizer_name, loss_fn):
    """
    Vanilla encoder-decoder CNN (no skip connections, no batch norm).

    Encoder     : `depth` x (Conv2D → MaxPool2D)
    Bottleneck  : one extra Conv2D at deepest spatial level
    Decoder     : `depth` x (UpSampling2D → Conv2D [+ Dropout])
    Output      : Conv2D(1, 1×1, sigmoid)

    Filter counts double each encoder stage and halve each decoder stage.
    """
    ks    = (kernel_size, kernel_size)
    model = Sequential(name="vanilla_cnn")

    # Encoder
    for stage in range(depth):
        filters = base_filters * (2 ** stage)
        if stage == 0:
            model.add(Conv2D(filters, ks, activation="relu", padding="same",
                             input_shape=(IMG_SIZE, IMG_SIZE, 3)))
        else:
            model.add(Conv2D(filters, ks, activation="relu", padding="same"))
        model.add(MaxPooling2D((2, 2)))

    # Bottleneck
    model.add(Conv2D(base_filters * (2 ** depth), ks, activation="relu", padding="same"))

    # Decoder
    for stage in range(depth - 1, -1, -1):
        filters = base_filters * (2 ** stage)
        model.add(UpSampling2D((2, 2)))
        model.add(Conv2D(filters, ks, activation="relu", padding="same"))
        if dropout_rate > 0.0:
            model.add(Dropout(dropout_rate))

    # Output
    model.add(Conv2D(1, (1, 1), activation="sigmoid", padding="same"))

    optimizer = (tf.keras.optimizers.Adam(learning_rate=learning_rate)
                 if optimizer_name == "adam"
                 else tf.keras.optimizers.RMSprop(learning_rate=learning_rate))

    model.compile(optimizer=optimizer, loss=loss_fn,
                  metrics=["accuracy", BinaryIoU(name="iou")])
    return model

print("✓ Model builder ready.")

In [ ]:
def find_mask_path(mask_folder, filename):
    """Return best-matching mask path for a given image filename."""
    stem, ext = os.path.splitext(filename)
    common_exts = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]
    candidate_stems = [stem]
    if "_sat_" in stem:
        candidate_stems.append(stem.replace("_sat_", "_mask_"))
    ext_order = [ext] + [e for e in common_exts if e != ext]
    for cstem in candidate_stems:
        for cext in ext_order:
            c = os.path.join(mask_folder, cstem + cext)
            if os.path.exists(c):
                return c
    return None

print("✓ find_mask_path helper ready.")

In [ ]:
print("Loading dataset …")
images_list, masks_list = [], []

for file in sorted(os.listdir(IMAGE_FOLDER)):
    img_path  = os.path.join(IMAGE_FOLDER, file)
    mask_path = find_mask_path(MASK_FOLDER, file)

    img = cv2.imread(img_path)
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0

    if mask_path is None:
        continue
    msk = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if msk is None:
        continue
    msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE)) / 255.0
    msk = np.expand_dims(msk, axis=-1)

    images_list.append(img)
    masks_list.append(msk)

images = np.array(images_list, dtype=np.float32)
masks  = np.array(masks_list,  dtype=np.float32)
print(f"  Loaded {len(images)} image-mask pairs  |  shape {images.shape}")

# 64 % train / 16 % val / 20 % test
X_tv, X_test, y_tv, y_test = train_test_split(
    images, masks, test_size=0.20, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.20, random_state=RANDOM_SEED)

print(f"  Train : {len(X_train)}  |  Val : {len(X_val)}  |  Test : {len(X_test)}")

In [ ]:
def sample_params():
    return {k: random.choice(v) for k, v in PARAM_GRID.items()}

def params_to_key(p):
    return (f"lr={p['learning_rate']:.0e}_bs={p['batch_size']}"
            f"_bf={p['base_filters']}_ks={p['kernel_size']}"
            f"_d={p['depth']}_dr={p['dropout_rate']}"
            f"_opt={p['optimizer_name']}_loss={p['loss_fn']}")

def run_trial(params):
    tf.keras.backend.clear_session()
    loss_fn = LOSS_MAP[params["loss_fn"]]
    model   = build_vanilla_cnn(
        base_filters   = params["base_filters"],
        kernel_size    = params["kernel_size"],
        depth          = params["depth"],
        dropout_rate   = params["dropout_rate"],
        learning_rate  = params["learning_rate"],
        optimizer_name = params["optimizer_name"],
        loss_fn        = loss_fn,
    )
    callbacks = [
        EarlyStopping(monitor="val_iou", mode="max", patience=5,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_iou", mode="max",
                          factor=0.5, patience=3, min_lr=1e-6, verbose=0),
    ]
    t0      = time.time()
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=MAX_EPOCHS, batch_size=params["batch_size"],
                        callbacks=callbacks, verbose=0)
    elapsed = time.time() - t0

    val_iou_h = history.history.get("val_iou", [0.0])
    best_idx  = int(np.argmax(val_iou_h))

    result = {
        **params,
        "best_val_iou":  round(float(max(val_iou_h)), 6),
        "best_val_acc":  round(float(history.history.get("val_accuracy", [0.0])[best_idx]), 6),
        "best_val_loss": round(float(history.history.get("val_loss",     [0.0])[best_idx]), 6),
        "best_epoch":    best_idx + 1,
        "elapsed_sec":   round(elapsed, 1),
        "total_params":  model.count_params(),
    }
    return result, model, history

print("✓ Trial helpers ready.")

In [ ]:
def compute_paper_metrics(model, X, y, threshold=0.5):
    """
    Aggregate TP/FP/FN/TN across the entire split, then compute
    Pixel Accuracy, IoU, Dice/F1, Precision, Recall, Specificity.
    Consistent with macro-aggregate reporting in segmentation papers.
    """
    y_pred = (model.predict(X, batch_size=16, verbose=0) >= threshold).astype(np.float32)
    y_true = y.astype(np.float32)
    yp, yt = y_pred.ravel(), y_true.ravel()

    TP  = np.sum(yt * yp)
    FP  = np.sum((1 - yt) * yp)
    FN  = np.sum(yt * (1 - yp))
    TN  = np.sum((1 - yt) * (1 - yp))
    eps = 1e-7

    return {
        "Pixel Accuracy": float((TP + TN) / (TP + TN + FP + FN + eps)),
        "IoU (Jaccard)":  float(TP / (TP + FP + FN + eps)),
        "Dice / F1":      float(2*TP / (2*TP + FP + FN + eps)),
        "Precision":      float(TP / (TP + FP + eps)),
        "Recall":         float(TP / (TP + FN + eps)),
        "Specificity":    float(TN / (TN + FP + eps)),
    }

print("✓ compute_paper_metrics ready.")

In [ ]:
csv_path        = os.path.join(OUTPUT_DIR, "hyperparam_search_results.csv")
best_model_path = os.path.join(OUTPUT_DIR, "best_vanilla_cnn.keras")

CSV_FIELDS = [
    "trial", "best_val_iou", "best_val_acc", "best_val_loss",
    "best_epoch", "elapsed_sec", "total_params",
    "learning_rate", "batch_size", "base_filters", "kernel_size",
    "depth", "dropout_rate", "optimizer_name", "loss_fn",
]

all_results  = []
best_val_iou = -1.0
best_model   = None
best_params  = None
seen_keys    = set()
histories    = {}

print("=" * 60)
print(f"  Starting random search  |  {N_TRIALS} trials  |  max {MAX_EPOCHS} epochs each")
print("=" * 60)

with open(csv_path, "w", newline="") as fcsv:
    writer = csv.DictWriter(fcsv, fieldnames=CSV_FIELDS)
    writer.writeheader()

    trial_num, attempts = 0, 0
    while trial_num < N_TRIALS and attempts < N_TRIALS * 5:
        attempts += 1
        params = sample_params()
        key    = params_to_key(params)
        if key in seen_keys:
            continue
        seen_keys.add(key)

        trial_num += 1
        print(f"Trial {trial_num:>2}/{N_TRIALS}  |  {key}")

        try:
            result, model, hist = run_trial(params)
        except Exception as exc:
            print(f"  ✗ Failed: {exc}")
            continue

        result["trial"] = trial_num
        all_results.append(result)
        histories[trial_num] = hist.history.get("val_iou", [])

        writer.writerow({k: result[k] for k in CSV_FIELDS})
        fcsv.flush()

        tag = ""
        if result["best_val_iou"] > best_val_iou:
            best_val_iou = result["best_val_iou"]
            best_params  = params
            best_model   = model
            model.save(best_model_path)
            tag = "  ← NEW BEST ★"

        print(f"  val_iou={result['best_val_iou']:.4f}  "
              f"val_acc={result['best_val_acc']:.4f}  "
              f"epoch={result['best_epoch']}  "
              f"t={result['elapsed_sec']}s{tag}\n")

print("Search complete.")

In [ ]:
best_hp_path = os.path.join(OUTPUT_DIR, "best_hyperparams.txt")

print("=" * 60)
print("  Evaluating best model on held-out test set …")
print("=" * 60)

keras_results = best_model.evaluate(X_test, y_test, verbose=0)
keras_metrics = dict(zip(best_model.metrics_names, keras_results))
paper_metrics = compute_paper_metrics(best_model, X_test, y_test)

print()
print("=" * 60)
print("  CONFERENCE TABLE  —  Vanilla CNN Baseline (Test Set)")
print("=" * 60)
print(f"  {'Metric':<24} {'Value':>8}")
print("  " + "-" * 34)
for name, val in paper_metrics.items():
    print(f"  {name:<24} {val:>8.4f}")
print("=" * 60)

# Sort leaderboard
all_results.sort(key=lambda x: x["best_val_iou"], reverse=True)
print("\n  Top-5 by val IoU:")
for rank, r in enumerate(all_results[:5], 1):
    print(f"  #{rank}  iou={r['best_val_iou']:.4f}  "
          f"lr={r['learning_rate']:.0e}  bs={r['batch_size']}  "
          f"bf={r['base_filters']}  d={r['depth']}  "
          f"loss={r['loss_fn']}  dr={r['dropout_rate']}")

# Save summary file
with open(best_hp_path, "w") as f:
    f.write("Vanilla CNN Baseline — Best Hyperparameters\n")
    f.write("=" * 46 + "\n\n")
    f.write("--- Hyperparameters ---\n")
    for k, v in best_params.items():
        f.write(f"{k:<22} {v}\n")
    f.write("\n--- Conference Table (Test Set) ---\n")
    f.write(f"{'Metric':<24} Value\n")
    f.write("-" * 34 + "\n")
    for name, val in paper_metrics.items():
        f.write(f"{name:<24} {val:.6f}\n")
    f.write("\n--- Keras built-in metrics ---\n")
    for name, val in keras_metrics.items():
        f.write(f"{name:<22} {val:.6f}\n")
    f.write("\n--- Search info ---\n")
    f.write(f"{'N_TRIALS':<22} {N_TRIALS}\n")
    f.write(f"{'MAX_EPOCHS':<22} {MAX_EPOCHS}\n")
    f.write(f"{'RANDOM_SEED':<22} {RANDOM_SEED}\n")
    f.write(f"{'Dataset size':<22} {len(images)}\n")
    f.write(f"{'Train/Val/Test':<22} {len(X_train)}/{len(X_val)}/{len(X_test)}\n")

print(f"\n  Saved: {best_hp_path}")
print(f"  Saved: {best_model_path}")
print(f"  Saved: {csv_path}")

In [ ]:
# ── Figure 1: leaderboard bar chart ──────────────────────────────────────────
top_n = min(10, len(all_results))
top   = all_results[:top_n]
labels = [f"#{i+1}\n{r['loss_fn']}\nlr={r['learning_rate']:.0e}"
          for i, r in enumerate(top)]
ious   = [r["best_val_iou"] for r in top]
dices  = [r["best_val_acc"] for r in top]   # proxy; real Dice is from paper_metrics

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Random Search — Top Configurations", fontsize=14, fontweight="bold")

ax = axes[0]
bars = ax.barh(range(top_n), ious, color="steelblue", edgecolor="white")
ax.set_yticks(range(top_n))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Val IoU")
ax.set_title("Top Configurations by Val IoU")
ax.invert_yaxis()
for bar, v in zip(bars, ious):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{v:.4f}", va="center", fontsize=8)

ax = axes[1]
# IoU history curves for top-5
colors = plt.cm.tab10.colors
for i, (tid, h) in enumerate(list(histories.items())[:5]):
    ax.plot(h, label=f"Trial {tid}", color=colors[i])
ax.set_xlabel("Epoch")
ax.set_ylabel("Val IoU")
ax.set_title("Val IoU Curves — Top-5 Trials")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "search_summary.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Figure 2: conference table bar chart ─────────────────────────────────────
metric_names  = list(paper_metrics.keys())
metric_values = list(paper_metrics.values())
colors_bar    = ["#2196F3","#4CAF50","#FF9800","#9C27B0","#F44336","#009688"]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(metric_names, metric_values, color=colors_bar, edgecolor="white", width=0.5)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("Vanilla CNN Baseline — Test Set Metrics", fontweight="bold")
for bar, v in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{v:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "conference_table.png"), dpi=150, bbox_inches="tight")
plt.show()

print("✓ Plots saved to Drive.")

In [ ]:
# Show 4 random test images with ground truth and best-model prediction
N_SHOW = 4
idx    = np.random.choice(len(X_test), N_SHOW, replace=False)
preds  = best_model.predict(X_test[idx], verbose=0)
preds_bin = (preds >= 0.5).astype(np.uint8)

fig, axes = plt.subplots(N_SHOW, 3, figsize=(12, 4 * N_SHOW))
fig.suptitle("Best Model Predictions on Test Set", fontsize=14, fontweight="bold")

for row, i in enumerate(idx):
    axes[row, 0].imshow(X_test[i])
    axes[row, 0].set_title("Satellite Image")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(y_test[i].squeeze(), cmap="Greens")
    axes[row, 1].set_title("Ground Truth Mask")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(preds_bin[row].squeeze(), cmap="Greens")
    axes[row, 2].set_title("Predicted Mask")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "predictions_sample.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ Prediction samples saved.")

In [ ]:
# Downloads the key output files to your local machine.
# (Files are already saved to Drive above; this is optional.)
from google.colab import files

for path in [best_model_path, csv_path, best_hp_path,
             os.path.join(OUTPUT_DIR, "search_summary.png"),
             os.path.join(OUTPUT_DIR, "conference_table.png")]:
    if os.path.isfile(path):
        files.download(path)
        print(f"  Downloaded: {os.path.basename(path)}")